<a href="https://www.kaggle.com/code/tiantengwang/t-leap-for-employee-hopping-ablation-study?scriptVersionId=339312731" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tiantengwang/employee-push-pull-moor-reason/balanced_undersampled_with_Push_Pull_Moor_Reasoning.xlsx
/kaggle/input/datasets/tiantengwang/employee-push-pull-moor-balanced/balanced_undersampled_with_Push_Pull_Moor.xlsx


In [3]:
df = pd.read_excel("/kaggle/input/datasets/tiantengwang/employee-push-pull-moor-reason/balanced_undersampled_with_Push_Pull_Moor_Reasoning.xlsx")
df.head(2)

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target,Push,Pull,Moor,Reasoning
0,30629,city_101,0.558,Male,Has relevent experience,no_enrollment,Graduate,STEM,8,10/49,Early Stage Startup,1,21,1.0,The profile does not explicitly report dissati...,"The candidate has relevant experience, 8 years...",The profile states that the last job change wa...,"The profile shows relevant experience, 8 years..."
1,23782,city_114,0.926,Male,No relevent experience,Full time course,High School,NaN,11,NaN,NaN,1,48,0.0,The profile does not explicitly report dissati...,The candidate has an expressed interest in the...,The profile states that the last job change wa...,"The profile shows no relevant experience, 11 y..."


In [5]:
df.columns

Index(['enrollee_id', 'city', 'city_development_index', 'gender',
       'relevent_experience', 'enrolled_university', 'education_level',
       'major_discipline', 'experience', 'company_size', 'company_type',
       'last_new_job', 'training_hours', 'target', 'Push', 'Pull', 'Moor',
       'Reasoning'],
      dtype='object')

In [9]:
df.shape

(9554, 16)

In [7]:
# 删除两列
df = df.drop(columns=['enrollee_id', 'city'])

# 重命名列
df = df.rename(columns={
    'target': 'FlagRepeatPurchase',
    'Push': 'Procedural',
    'Pull': 'Financial',
    'Moor': 'Relational'
})

# 查看结果
print(df.columns)

Index(['city_development_index', 'gender', 'relevent_experience',
       'enrolled_university', 'education_level', 'major_discipline',
       'experience', 'company_size', 'company_type', 'last_new_job',
       'training_hours', 'FlagRepeatPurchase', 'Procedural', 'Financial',
       'Relational', 'Reasoning'],
      dtype='object')


In [8]:
import pandas as pd

SEED = 42

# 分别抽取1000条
df_0 = df[df["FlagRepeatPurchase"] == 0].sample(
    n=1000,
    random_state=SEED
)

df_1 = df[df["FlagRepeatPurchase"] == 1].sample(
    n=1000,
    random_state=SEED
)

# 合并
df_balanced = pd.concat([df_0, df_1], axis=0)

# 打乱顺序
df_balanced = df_balanced.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# 查看结果
print(df_balanced["FlagRepeatPurchase"].value_counts())

print(df_balanced.shape)

FlagRepeatPurchase
1.0    1000
0.0    1000
Name: count, dtype: int64
(2000, 16)


In [10]:
# 每条文本的token数量（按空格）
token_counts = df["Reasoning"].fillna("").str.split().str.len()

print("Maximum tokens:", token_counts.max())

Maximum tokens: 152


## 1.1 构建数据集及获得embeddings

In [11]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM
from tensorflow.keras.layers import Dense, Dropout

# =====================================================
# Load Data
# =====================================================

# 如果已经有df，这一行不用改
df = df_balanced

# =====================================================
# Prepare Data
# =====================================================

# X = df.drop(columns=["FlagRepeatPurchase"]).copy()
X = df.drop(columns=["FlagRepeatPurchase", "Procedural", "Financial", "Relational", "Reasoning"]).copy()

y = (
    df["FlagRepeatPurchase"]
    .astype(int)
    .values
)

# Encode categorical variables
for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

X = X.values

In [12]:
X.shape

(2000, 11)

In [13]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from transformers import BertTokenizer, BertModel
from tqdm import tqdm

# =====================================================
# Load Data
# =====================================================

# df = df

# =====================================================
# BERT
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

bert = BertModel.from_pretrained("bert-base-uncased")

bert.to(device)
bert.eval()

# =====================================================
# Encode Function
# =====================================================

@torch.no_grad()
def encode_texts(texts,
                 max_length=128,
                 batch_size=64):
    """
    Encode a list of texts into CLS embeddings.

    Returns
    -------
    embeddings : numpy array
        shape = (N,768)
    """

    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):

        batch = texts[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            k:v.to(device)
            for k,v in encoded.items()
        }

        outputs = bert(**encoded)

        cls_embedding = outputs.last_hidden_state.mean(dim=1)

        all_embeddings.append(
            cls_embedding.cpu().numpy()
        )

    embeddings = np.vstack(all_embeddings)

    return embeddings

# =====================================================
# Encode Procedural
# =====================================================

proc_embeddings = encode_texts(
    df["Procedural"].fillna("").astype(str).tolist()
)

print(proc_embeddings.shape)

# =====================================================
# Encode Financial
# =====================================================

fin_embeddings = encode_texts(
    df["Financial"].fillna("").astype(str).tolist()
)

print(fin_embeddings.shape)

# =====================================================
# Encode Relational
# =====================================================

rel_embeddings = encode_texts(
    df["Relational"].fillna("").astype(str).tolist()
)

print(rel_embeddings.shape)

# =====================================================
# Encode Reasoning
# =====================================================

rea_embeddings = encode_texts(
    df["Reasoning"].fillna("").astype(str).tolist()
)

print(rea_embeddings.shape)

# =====================================================
# Save
# =====================================================

np.save("proc_embeddings.npy", proc_embeddings)

np.save("fin_embeddings.npy", fin_embeddings)

np.save("rel_embeddings.npy", rel_embeddings)

np.save("rea_embeddings.npy", rel_embeddings)

print("\nSaved successfully.")

print("Procedural :", proc_embeddings.shape)
print("Financial  :", fin_embeddings.shape)
print("Relational :", rel_embeddings.shape)
print("Reasoning :", rea_embeddings.shape)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

(2000, 768)


100%|██████████| 32/32 [00:11<00:00,  2.84it/s]


(2000, 768)


100%|██████████| 32/32 [00:15<00:00,  2.12it/s]

(2000, 768)

Saved successfully.
Procedural : (2000, 768)
Financial  : (2000, 768)
Relational : (2000, 768)
Reasoning : (2000, 768)


In [11]:
proc_embeddings = np.load("proc_embeddings.npy")
fin_embeddings  = np.load("fin_embeddings.npy")
rel_embeddings  = np.load("rel_embeddings.npy")

## 1.2 定义原始数据集+增强的数据集+无理论增强的数据集

In [14]:

# =====================================================
# Dataset
# =====================================================

from torch.utils.data import Dataset, DataLoader

class StructuredDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, index):

        return self.X[index], self.y[index]


class AugmentedDataset(Dataset):

    def __init__(
        self,
        X,
        proc,
        fin,
        rel,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.proc = torch.tensor(
            proc,
            dtype=torch.float32
        )

        self.fin = torch.tensor(
            fin,
            dtype=torch.float32
        )

        self.rel = torch.tensor(
            rel,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, index):

        return (
            self.X[index],
            self.proc[index],
            self.fin[index],
            self.rel[index],
            self.y[index]
        )

class AugmentedDatasetNoTheory(Dataset):

    def __init__(
        self,
        X,
        rea,
        y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.rea = torch.tensor(
            rea,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, index):

        return (
            self.X[index],
            self.rea[index],
            self.y[index]
        )

## 1.3 定义模型

In [31]:
import torch
import torch.nn as nn

# =====================================================
# Model 2: Bi-LSTM
# # =====================================================

# class BiLSTMClassifier(nn.Module):

#     def __init__(self, structured_dim):

#         super().__init__()

#         self.encoder = nn.Sequential(

#             nn.Linear(structured_dim, 128),

#             nn.ReLU(),

#             nn.Dropout(0.3),
#             nn.Linear(128, 768)


#         )

#         self.classifier = nn.Sequential(

#             nn.Linear(768, 128),

#             nn.ReLU(),

#             nn.Dropout(0.3),

#             nn.Linear(128, 2)

#         )

#     def forward(self, x):

#         x = self.encoder(x)

#         return self.classifier(x)
        
class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        hidden_dim=64
    ):

        super().__init__()

        self.bilstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        # self.fc1 = nn.Linear(
        #     hidden_dim * 2,
        #     16
        # )

        

        # self.fc2 = nn.Linear(
        #     16,
        #     2
        # )
        self.fc1 = nn.Linear(
             hidden_dim*2,
                  64
              )
        
        self.fc2 = nn.Linear(
                  64,
              16
             )
        
        self.fc3 = nn.Linear(
              16,
               2
             )
        
        self.relu = nn.ReLU()

    def forward(self, x):

        # x: (batch, number_of_features)
        x = x.unsqueeze(-1)

        # x: (batch, number_of_features, 1)
        output, _ = self.bilstm(x)

 #       x = output[:, -1, :]

        x = output.mean(dim=1)

        # x = self.dropout(x)
        # x = self.relu(self.fc1(x))

        x = self.dropout(x) 

        x = self.relu(self.fc1(x))

        x = self.dropout(x)

        x = self.relu(self.fc2(x))

        return self.fc3(x)

 #      return self.fc2(x)


# =====================================================
# Model 3: BERT Embeddings + Bi-LSTM
# =====================================================


class BERTBiLSTMClassifier(nn.Module):

    def __init__(self, structured_dim):

        super().__init__()

        self.structured_projection = nn.Sequential(

            nn.Linear(structured_dim, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 768)

        )

        # self.shortcut = nn.Linear(
        #     structured_dim,
        #     768
        # )

 #       self.norm = nn.LayerNorm(768)

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Sequential(

            nn.Linear(768 * 4, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 2),


        )

    def forward(
        self,
        structured,
        proc,
        fin,
        rel
    ):

   #     shortcut = self.shortcut(structured)

        structured = self.structured_projection(
            structured
        )

  #      structured = structured + shortcut

 #      structured = self.norm(


        x = torch.cat(

            [

                structured,

                proc,

                fin,

                rel

            ],

            dim=1

        )

        x = self.dropout(x)

        return self.classifier(x)

### 定义模型8

In [35]:
class BERTBiLSTMClassifier_no_theory(nn.Module):

    def __init__(self, structured_dim):

        super().__init__()

        self.structured_projection = nn.Sequential(

            nn.Linear(structured_dim, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 768)

        )

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Sequential(

            nn.Linear(1536, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 2),


        )

    def forward(
        self,
        structured,
        rea
    ):

   #     shortcut = self.shortcut(structured)

        structured = self.structured_projection(
            structured
        )

  #      structured = structured + shortcut

 #      structured = self.norm(


        x = torch.cat(

            [

                structured,

                rea

            ],

            dim=1

        )

        x = self.dropout(x)

        return self.classifier(x)

In [30]:
class BERTBiLSTMClassifier(nn.Module):

    def __init__(self, structured_dim):

        super().__init__()

        self.structured_projection = nn.Sequential(

            nn.Linear(structured_dim, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 768)

        )

        # self.shortcut = nn.Linear(
        #     structured_dim,
        #     768
        # )

 #       self.norm = nn.LayerNorm(768)

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Sequential(

            nn.Linear(768 * 4, 128),

            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(128, 2),


        )

    def forward(
        self,
        structured,
        proc,
        fin,
        rel
    ):

   #     shortcut = self.shortcut(structured)

        structured = self.structured_projection(
            structured
        )

  #      structured = structured + shortcut

 #      structured = self.norm(


        x = torch.cat(

            [

                structured,

                proc,

                fin,

                rel

            ],

            dim=1

        )

        x = self.dropout(x)

        return self.classifier(x)

In [29]:
class Model4(nn.Module):

    def __init__(
        self,
        structured_dim=14,
        bert_dim=768,
        hidden_dim=128,
        num_heads=4,
        dropout=0.3
    ):

        super().__init__()

        ############################################################
        # Structured Projection
        ############################################################

        self.structured_projection = nn.Sequential(

            nn.Linear(structured_dim, 64),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(64, hidden_dim),

            nn.LayerNorm(hidden_dim)

        )

        ############################################################
        # Shared Theory Projection
        ############################################################

        self.theory_projection = nn.Sequential(

            nn.Linear(bert_dim, 256),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(256, hidden_dim),

            nn.LayerNorm(hidden_dim)

        )

        ############################################################
        # Cross Attention
        ############################################################

        self.cross_attention = nn.MultiheadAttention(

            embed_dim=hidden_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True

        )

        # ############################################################
        # # Transformer Feed Forward Network 在cross-attention之后加入FFN 学习更好的T
        # ############################################################

        # self.ffn = nn.Sequential(

        #     nn.Linear(hidden_dim, hidden_dim * 2),

        #     nn.GELU(),

        #     nn.Dropout(dropout),

        #     nn.Linear(hidden_dim * 2, hidden_dim),

        #     nn.Dropout(dropout)

        # )

        # ############################################################
        # # LayerNorm after Attention
        # ############################################################

        # self.attn_norm = nn.LayerNorm(hidden_dim) 

        # ############################################################
        # # LayerNorm after FFN 进行归一化
        # ############################################################

        # self.ffn_norm = nn.LayerNorm(hidden_dim)

        ############################################################
        # Classifier
        ############################################################

        self.classifier = nn.Sequential(

            nn.Linear(hidden_dim * 2, 64),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(64, 16),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(16, 2)

        )

    ############################################################
    # Forward
    ############################################################

    def forward(

        self,

        structured,

        proc,

        fin,

        rel

    ):

        ############################################################
        # Structured Projection
        ############################################################

        S = self.structured_projection(structured)

        ############################################################
        # Shared Theory Projection
        ############################################################

        P = self.theory_projection(proc)

        F = self.theory_projection(fin)

        R = self.theory_projection(rel)

        ############################################################
        # Query
        ############################################################

        Q = S.unsqueeze(1)

        ############################################################
        # Key / Value
        ############################################################

        KV = torch.stack(

            [

                P,

                F,

                R

            ],

            dim=1

        )

        ############################################################
        # Cross Attention
        ############################################################

        T, attention_weights = self.cross_attention(

            query=Q,

            key=KV,

            value=KV

        )

        ############################################################
        # remove sequence dimension
        ############################################################

        T = T.squeeze(1)

        # ############################################################
        # # Cross Attention
        # ############################################################

        # T, _ = self.cross_attention(

        #     query=Q,

        #     key=KV,

        #     value=KV

        # )

        # T = T.squeeze(1)

        # ############################################################
        # # Residual + LayerNorm
        # ############################################################

        # T = self.attn_norm(

        #     T + S

        # )

        # ############################################################
        # # Feed Forward Network
        # ############################################################

        # T_ffn = self.ffn(T)

        # ############################################################
        # # Residual + LayerNorm
        # ############################################################

        # T = self.ffn_norm(

        #     T + T_ffn

        # )

        ############################################################
        # Fusion
        ############################################################

        fusion = torch.cat(

            [

                S,

                T

            ],

            dim=1

        )

        ############################################################
        # Prediction
        ############################################################

        logits = self.classifier(fusion)

        # if return_attention:
        #     return logits, attention_weights
        
       # return logits
        return logits

In [28]:
class Model5(nn.Module):

    def __init__(
        self,
        structured_dim=16,
        bert_dim=768,
        hidden_dim=128,
        num_heads=4,
        dropout=0.3
    ):

        super().__init__()

        ############################################################
        # Structured Projection
        ############################################################

        self.structured_projection = nn.Sequential(

            nn.Linear(structured_dim, 64),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(64, hidden_dim),

            nn.LayerNorm(hidden_dim)

        )

        ############################################################
        # Shared Theory Projection
        ############################################################

        self.theory_projection = nn.Sequential(

            nn.Linear(bert_dim, 256),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(256, hidden_dim),

            nn.LayerNorm(hidden_dim)

        )

        ############################################################
        # Cross Attention
        ############################################################

        self.cross_attention = nn.MultiheadAttention(

            embed_dim=hidden_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True

        )

        # ############################################################
        # # Transformer Feed Forward Network 在cross-attention之后加入FFN 学习更好的T
        # ############################################################

        # self.ffn = nn.Sequential(

        #     nn.Linear(hidden_dim, hidden_dim * 2),

        #     nn.GELU(),

        #     nn.Dropout(dropout),

        #     nn.Linear(hidden_dim * 2, hidden_dim),

        #     nn.Dropout(dropout)

        # )

        # ############################################################
        # # LayerNorm after Attention
        # ############################################################

        # self.attn_norm = nn.LayerNorm(hidden_dim) 

        # ############################################################
        # # LayerNorm after FFN 进行归一化
        # ############################################################

        # self.ffn_norm = nn.LayerNorm(hidden_dim)

        ############################################################
        # Classifier
        ############################################################

        self.classifier = nn.Sequential(

            nn.Linear(hidden_dim * 2, 64),

            nn.ReLU(),

            nn.Dropout(dropout),

            nn.Linear(64, 16),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(16, 2)

        )

    ############################################################
    # Forward
    ############################################################

    def forward(

        self,

        structured,

        proc,

        fin,

        rel

    ):

        ############################################################
        # Structured Projection
        ############################################################

        S = self.structured_projection(structured)

        ############################################################
        # Shared Theory Projection
        ############################################################

        P = self.theory_projection(proc)

        F = self.theory_projection(fin)

        R = self.theory_projection(rel)

        ############################################################
        # Query
        ############################################################

        Q = S.unsqueeze(1)

        ############################################################
        # Key / Value
        ############################################################

        KV = torch.stack(

            [

                P,

                F,

                R

            ],

            dim=1

        )

        ############################################################
        # Cross Attention
        ############################################################

        T, attention_weights = self.cross_attention(

            query=Q,

            key=KV,

            value=KV

        )

        ############################################################
        # remove sequence dimension
        ############################################################

        T = T.squeeze(1)

        # ############################################################
        # # Cross Attention
        # ############################################################

        # T, _ = self.cross_attention(

        #     query=Q,

        #     key=KV,

        #     value=KV

        # )

        # T = T.squeeze(1)

        # ############################################################
        # # Residual + LayerNorm
        # ############################################################

        # T = self.attn_norm(

        #     T + S

        # )

        # ############################################################
        # # Feed Forward Network
        # ############################################################

        # T_ffn = self.ffn(T)

        # ############################################################
        # # Residual + LayerNorm
        # ############################################################

        # T = self.ffn_norm(

        #     T + T_ffn

        # )

        ############################################################
        # Fusion
        ############################################################

        fusion = torch.cat(

            [

                S,

                T

            ],

            dim=1

        )

        ############################################################
        # Prediction
        ############################################################

        logits = self.classifier(fusion)

        # if return_attention:
        #     return logits, attention_weights
        
       # return logits
        return logits, S, T

# Syn + CL loss definition

In [27]:
# 定义协同约束的loss

import torch
import torch.nn as nn
import torch.nn.functional as F


class AlignmentLoss(nn.Module):

    def forward(self,S,T):

        S = F.normalize(
            S,
            dim=1
        )

        T = F.normalize(
            T,
            dim=1
        )

        cosine = (
            S*T
        ).sum(
            dim=1
        )

        return (
            1-cosine
        ).mean()

SEED = 42

# 定义对比学习的loss

import torch.nn.functional as F

class CrossViewInfoNCELoss(nn.Module):

    def __init__(self, temperature=0.5):

        super().__init__()

        self.temperature = temperature

    def forward(
        self,
        S,
        T,
        labels
    ):

        S = F.normalize(S, dim=1)
        T = F.normalize(T, dim=1)

        logits = torch.matmul(
            S,
            T.T
        ) / self.temperature

        labels = labels.view(-1,1)

        mask = (
            labels == labels.T
        ).float()

        log_prob = F.log_softmax(
            logits,
            dim=1
        )

        loss = -(
            mask * log_prob
        ).sum(1) / mask.sum(1)

        return loss.mean()


LAMBDA_CONTRAST = 0.01

## Training and testing

## model 2 (FFNN)， 3 (FFNN + BERT), 4 (Thoery alginment) 的训练和测试

In [26]:
# =====================================================
# Training and Evaluation Functions
# =====================================================

def train_structured_model(
    model,
    train_loader,
    epochs=30
):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,# 原来的： 5e-4
        weight_decay=1e-4
    )

    model.train()

    for epoch in range(epochs):

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(
                logits,
                y_batch
            )

            loss.backward()
            optimizer.step()


def evaluate_structured_model(
    model,
    test_loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_score.extend(
                probabilities.cpu().numpy()
            )

    return (
        roc_auc_score(y_true, y_score),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    )


# for BERT+MLP
def train_augmented_model(
    model,
    train_loader,
    epochs=10
):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    model.train()

    for epoch in range(epochs):

        for batch in train_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(
                X_batch,
                proc,
                fin,
                rel
            )

            loss = criterion(
                logits,
                y_batch
            )

            loss.backward()
            optimizer.step()


def evaluate_augmented_model(
    model,
    test_loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for batch in test_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)

            logits = model(
                X_batch,
                proc,
                fin,
                rel
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_score.extend(
                probabilities.cpu().numpy()
            )

    return (
        roc_auc_score(y_true, y_score),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    )


## model 8 的training and testing

In [23]:
# for BERT+MLP
def train_augmented_model_no_theory(
    model,
    train_loader,
    epochs=10
):

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    model.train()

    for epoch in range(epochs):

        for batch in train_loader:

            X_batch, rea, y_batch = batch

            X_batch = X_batch.to(device)

            rea = rea.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits = model(
                X_batch,
                rea
            )

            loss = criterion(
                logits,
                y_batch
            )

            loss.backward()
            optimizer.step()


def evaluate_augmented_model_no_theory(
    model,
    test_loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for batch in test_loader:

            X_batch, rea, y_batch = batch

            X_batch = X_batch.to(device)
            rea = rea.to(device)

            logits = model(
                X_batch,
                rea
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_score.extend(
                probabilities.cpu().numpy()
            )

    return (
        roc_auc_score(y_true, y_score),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    )

## model 5 (cl), 6 (syn) 的训练 and 测试 

In [25]:
# 定义新的 训练 测试的函数

def train_augmented_model_contrastive(

    model,

    train_loader,

    epochs=30,

    lambda_con=LAMBDA_CONTRAST

):

    criterion = nn.CrossEntropyLoss()

    contrastive_loss = CrossViewInfoNCELoss()

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=1e-4,

        weight_decay=1e-4

    )

    model.train()

    for epoch in range(epochs):

        for batch in train_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits, S, T = model(

                X_batch,

                proc,

                fin,

                rel

            )

            loss_cls = criterion(

                logits,

                y_batch

            )

            loss_con = contrastive_loss(

                S,

                T,

                y_batch

            )

            loss = (

                loss_cls

                +

                lambda_con * loss_con

            )

            loss.backward()

            optimizer.step()

def evaluate_augmented_model_contrastive(
    model,
    test_loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for batch in test_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)

            logits, _, _ = model(

                  X_batch,

                    proc,

                     fin,

                    rel

                   )

            # logits = model(
            #     X_batch,
            #     proc,
            #     fin,
            #     rel
            # )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_score.extend(
                probabilities.cpu().numpy()
            )

    return (
        roc_auc_score(y_true, y_score),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    )

def train_augmented_model_algin(
    model,
    train_loader,
    epochs=30,
    lambda_align=0.1
):

    criterion = nn.CrossEntropyLoss()

    alignment_criterion = AlignmentLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    model.train()

    for epoch in range(epochs):

        for batch in train_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            ####################################################
            # Forward
            ####################################################

            logits, S, T = model(
                X_batch,
                proc,
                fin,
                rel
            )

            ####################################################
            # Cross Entropy
            ####################################################

            loss_ce = criterion(
                logits,
                y_batch
            )

            ####################################################
            # Alignment Loss
            ####################################################

            loss_align = alignment_criterion(
                S,
                T
            )

            ####################################################
            # Total Loss
            ####################################################

            loss = (
                loss_ce
                + lambda_align * loss_align
            )

            loss.backward()

            optimizer.step()
        

def evaluate_augmented_model_algin(
    model,
    test_loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for batch in test_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)

            ####################################################
            # Forward
            ####################################################

            logits, _, _ = model(
                X_batch,
                proc,
                fin,
                rel
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_score.extend(
                probabilities.cpu().numpy()
            )

    return (
        roc_auc_score(y_true, y_score),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    )



## model 7 (syn + cl) 的训练和测试

In [33]:
# =====================================================
# Model 7:
# Theory Cross Attention
# + Contrastive
# + Alignment
# =====================================================

def train_augmented_model_contrastive_align(
    model,
    train_loader,
    epochs=30,
    lambda_con=LAMBDA_CONTRAST,
    lambda_align=0.1
):

    criterion = nn.CrossEntropyLoss()

    contrastive_loss = CrossViewInfoNCELoss()

    alignment_loss = AlignmentLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    model.train()

    for epoch in range(epochs):

        for batch in train_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            logits, S, T = model(
                X_batch,
                proc,
                fin,
                rel
            )

            loss_cls = criterion(
                logits,
                y_batch
            )

            loss_con = contrastive_loss(
                S,
                T,
                y_batch
            )

            loss_align = alignment_loss(
                S,
                T
            )

            loss = (
                loss_cls
                + lambda_con * loss_con
                + lambda_align * loss_align
            )

            loss.backward()

            optimizer.step()


def evaluate_augmented_model_contrastive_align(
    model,
    test_loader
):

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():

        for batch in test_loader:

            X_batch, proc, fin, rel, y_batch = batch

            X_batch = X_batch.to(device)
            proc = proc.to(device)
            fin = fin.to(device)
            rel = rel.to(device)

            logits, _, _ = model(
                X_batch,
                proc,
                fin,
                rel
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                logits,
                dim=1
            )

            y_true.extend(
                y_batch.numpy()
            )

            y_pred.extend(
                predictions.cpu().numpy()
            )

            y_score.extend(
                probabilities.cpu().numpy()
            )

    return (
        roc_auc_score(y_true, y_score),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    )

## 交叉验证, 打印test 中的结果

In [43]:

# =====================================================
# Cross Validation
# =====================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

xgb_auc, xgb_acc, xgb_f1 = [], [], []
lstm_auc, lstm_acc, lstm_f1 = [], [], []
bert_auc, bert_acc, bert_f1 = [], [], []
model4_auc, model4_acc, model4_f1 = [], [], []
model5_auc=[]
model5_acc=[]
model5_f1=[]

model6_auc=[]
model6_acc=[]
model6_f1=[]

model7_auc = []
model7_acc = []
model7_f1 = []

model8_auc = []
model8_acc = []
model8_f1 = []


for fold, (train_idx, test_idx) in enumerate(
    skf.split(X, y),
    1
):

    print(f"\nFold {fold}")

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Standardization只在训练折上拟合
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    ).astype(np.float32)

    X_test_scaled = scaler.transform(
        X_test
    ).astype(np.float32)


    # =================================================
    # Model 1: XGBoost
    # =================================================

    xgb = XGBClassifier(
        n_estimators=5,
        max_depth=2,
        learning_rate=0.0005,
        subsample=0.4,
        colsample_bytree=0.4,
        eval_metric="logloss",
        random_state=SEED
    )

    xgb.fit(
        X_train,
        y_train
    )

    probabilities = xgb.predict_proba(
        X_test
    )[:, 1]

    predictions = (
        probabilities > 0.5
    ).astype(int)

    xgb_auc.append(
        roc_auc_score(
            y_test,
            probabilities
        )
    )

    xgb_acc.append(
        accuracy_score(
            y_test,
            predictions
        )
    )

    xgb_f1.append(
        f1_score(
            y_test,
            predictions
        )
    )


    # =================================================
    # Model 2: FFNN
    # =================================================

    train_dataset = StructuredDataset(
        X_train_scaled,
        y_train
    )

    test_dataset = StructuredDataset(
        X_test_scaled,
        y_test
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False
    )

    torch.manual_seed(SEED + fold)

    model2 = BiLSTMClassifier().to(device)

    train_structured_model(
        model2,
        train_loader,
        epochs=60
    )

    auc, acc, f1 = evaluate_structured_model(
        model2,
        test_loader
    )

    lstm_auc.append(auc)
    lstm_acc.append(acc)
    lstm_f1.append(f1)


    # =================================================
    # Model 3: BERT + Bi-LSTM
    # =================================================

    train_dataset = AugmentedDataset(
        X_train_scaled,
        proc_embeddings[train_idx],
        fin_embeddings[train_idx],
        rel_embeddings[train_idx],
        y_train
    )

    test_dataset = AugmentedDataset(
        X_test_scaled,
        proc_embeddings[test_idx],
        fin_embeddings[test_idx],
        rel_embeddings[test_idx],
        y_test
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False
    )

    torch.manual_seed(SEED + fold)

    model3 = BERTBiLSTMClassifier(
        structured_dim=X_train_scaled.shape[1]
    ).to(device)

    train_augmented_model(
        model3,
        train_loader,
        epochs=1
    )

    auc, acc, f1 = evaluate_augmented_model(
        model3,
        test_loader
    )

    bert_auc.append(auc)
    bert_acc.append(acc)
    bert_f1.append(f1)

    # =================================================
    # Model 4: Theory Cross Attention
    # =================================================

    torch.manual_seed(SEED + fold)

    model4 = Model4(
        structured_dim=X_train_scaled.shape[1]
    ).to(device)

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False
    )

    
    train_augmented_model(
        model4,
        train_loader,
        epochs=5
    )

    auc, acc, f1 = evaluate_augmented_model(
        model4,
        test_loader
    )

    model4_auc.append(auc)
    model4_acc.append(acc)
    model4_f1.append(f1)


    #################################################
    # Model 5
   #################################################

    torch.manual_seed(SEED + fold)

    model5 = Model5(structured_dim=X_train_scaled.shape[1]

          ).to(device)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False
    )

    train_augmented_model_contrastive( model5,

    train_loader,

    epochs=30, 
)

    auc, acc, f1 = evaluate_augmented_model_contrastive(

    model5,

    test_loader

)

    model5_auc.append(auc)

    model5_acc.append(acc)

    model5_f1.append(f1)

    # =================================================
    # Model 6: Theory Cross Attention + algin
    # =================================================

    torch.manual_seed(SEED + fold)

    model6 = Model5(
        structured_dim=X_train_scaled.shape[1]
    ).to(device)

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=64,
        shuffle=False
    )

    train_augmented_model_algin(
        model6,
        train_loader,
        # batch_size=128,
        # shuffle=True,
        epochs=30
    )

    auc, acc, f1 = evaluate_augmented_model_algin(
        model6,
        test_loader
    )

    model6_auc.append(auc)
    model6_acc.append(acc)
    model6_f1.append(f1)

    
    # =================================================
    # Model 7: Theory Cross Attention + algin + CL
    # =================================================

    
    torch.manual_seed(SEED + fold)

    model7 = Model5(
    structured_dim=X_train_scaled.shape[1]
).to(device)

    train_augmented_model_contrastive_align(
    model7,
    train_loader,
    epochs=30
)

    auc, acc, f1 = evaluate_augmented_model_contrastive_align(
    model7,
    test_loader
)

    model7_auc.append(auc)
    model7_acc.append(acc)
    model7_f1.append(f1)
    
    # =================================================
    # Model 8: BERT + Bi-LSTM (no theory)
    # =================================================

    train_dataset = AugmentedDatasetNoTheory(
        X_train_scaled,
        rea_embeddings[train_idx],
        y_train
    )

    test_dataset = AugmentedDatasetNoTheory(
        X_test_scaled,
        rea_embeddings[test_idx],
        y_test
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False
    )

    torch.manual_seed(SEED + fold)

    model3 = BERTBiLSTMClassifier_no_theory(
        structured_dim=X_train_scaled.shape[1]
    ).to(device)

    train_augmented_model_no_theory(
        model3,
        train_loader,
        epochs=1
    )

    auc, acc, f1 = evaluate_augmented_model_no_theory(
        model3,
        test_loader
    )

    model8_auc.append(auc)
    model8_acc.append(acc)
    model8_f1.append(f1)


    print(

    f"XGBoost AUC: {xgb_auc[-1]:.4f} | "

    f"FFNN AUC: {lstm_auc[-1]:.4f} | "

    f"BERT+FFNN AUC: {bert_auc[-1]:.4f} | "

    f"CrossAttn AUC: {model4_auc[-1]:.4f} | "

    f"CrossAttn+CL AUC: {model5_auc[-1]:.4f} | "

    f"CrossAttn+algin AUC: {model6_auc[-1]:.4f} | "
    
    f"CrossAttn+algin+CL AUC: {model7_auc[-1]:.4f} | "
    
    f"T-LEAP w/o theory AUC: {model8_auc[-1]:.4f} | "

)



results = pd.DataFrame({

    "Model": [

        "XGBoost",
        "T-LEAP w/o Prompt",
        "T-LEAP w/o Align(ment)-cross-attention fusion",
        "T-LEAP w/o (CL+Syn)",
        "T-LEAP w/o (Syn)",
        "T-LEAP w/o (CL)",
        "T-LEAP",
        "T-LEAP w/o theory",

    ],

    "AUC": [

        f"{np.mean(xgb_auc):.4f} ± {np.std(xgb_auc):.4f}",
        f"{np.mean(lstm_auc):.4f} ± {np.std(lstm_auc):.4f}",
        f"{np.mean(bert_auc):.4f} ± {np.std(bert_auc):.4f}",
        f"{np.mean(model4_auc):.4f} ± {np.std(model4_auc):.4f}",
        f"{np.mean(model5_auc):.4f} ± {np.std(model5_auc):.4f}",
        f"{np.mean(model6_auc):.4f} ± {np.std(model6_auc):.4f}",
        f"{np.mean(model7_auc):.4f} ± {np.std(model7_auc):.4f}",
        f"{np.mean(model8_auc):.4f} ± {np.std(model8_auc):.4f}"

    ],

    "Accuracy": [

        f"{np.mean(xgb_acc):.4f} ± {np.std(xgb_acc):.4f}",
        f"{np.mean(lstm_acc):.4f} ± {np.std(lstm_acc):.4f}",
        f"{np.mean(bert_acc):.4f} ± {np.std(bert_acc):.4f}",
        f"{np.mean(model4_acc):.4f} ± {np.std(model4_acc):.4f}",
        f"{np.mean(model5_acc):.4f} ± {np.std(model5_acc):.4f}",
        f"{np.mean(model6_acc):.4f} ± {np.std(model6_acc):.4f}",
        f"{np.mean(model7_acc):.4f} ± {np.std(model7_acc):.4f}",
        f"{np.mean(model8_acc):.4f} ± {np.std(model8_acc):.4f}"

    ],

    "F1": [

        f"{np.mean(xgb_f1):.4f} ± {np.std(xgb_f1):.4f}",
        f"{np.mean(lstm_f1):.4f} ± {np.std(lstm_f1):.4f}",
        f"{np.mean(bert_f1):.4f} ± {np.std(bert_f1):.4f}",
        f"{np.mean(model4_f1):.4f} ± {np.std(model4_f1):.4f}",
        f"{np.mean(model5_f1):.4f} ± {np.std(model5_f1):.4f}",
        f"{np.mean(model6_f1):.4f} ± {np.std(model6_f1):.4f}",
        f"{np.mean(model7_f1):.4f} ± {np.std(model7_f1):.4f}",
        f"{np.mean(model8_f1):.4f} ± {np.std(model8_f1):.4f}"
        

    ]

})

print("\n")
print("=" * 75)
print("5-Fold Cross Validation Results")
print("=" * 75)
print(results.to_string(index=False))


Fold 1
XGBoost AUC: 0.7452 | FFNN AUC: 0.6852 | BERT+FFNN AUC: 0.7733 | CrossAttn AUC: 0.7908 | CrossAttn+CL AUC: 0.8011 | CrossAttn+algin AUC: 0.8036 | CrossAttn+algin+CL AUC: 0.8035 | T-LEAP w/o theory AUC: 0.7602 | 

Fold 2
XGBoost AUC: 0.7492 | FFNN AUC: 0.6709 | BERT+FFNN AUC: 0.7401 | CrossAttn AUC: 0.7375 | CrossAttn+CL AUC: 0.7898 | CrossAttn+algin AUC: 0.7907 | CrossAttn+algin+CL AUC: 0.7912 | T-LEAP w/o theory AUC: 0.7207 | 

Fold 3
XGBoost AUC: 0.7498 | FFNN AUC: 0.7076 | BERT+FFNN AUC: 0.7359 | CrossAttn AUC: 0.7408 | CrossAttn+CL AUC: 0.7776 | CrossAttn+algin AUC: 0.7780 | CrossAttn+algin+CL AUC: 0.7783 | T-LEAP w/o theory AUC: 0.7302 | 

Fold 4
XGBoost AUC: 0.7658 | FFNN AUC: 0.6879 | BERT+FFNN AUC: 0.7534 | CrossAttn AUC: 0.7629 | CrossAttn+CL AUC: 0.7812 | CrossAttn+algin AUC: 0.7838 | CrossAttn+algin+CL AUC: 0.7837 | T-LEAP w/o theory AUC: 0.7483 | 

Fold 5
XGBoost AUC: 0.7834 | FFNN AUC: 0.7197 | BERT+FFNN AUC: 0.8007 | CrossAttn AUC: 0.7942 | CrossAttn+CL AUC: 0.800

In [44]:
results_ = pd.DataFrame({

    "Model": [

        "XGBoost",
        "T-LEAP w/o Prompt",
        "T-LEAP w/o Align(ment)-cross-attention fusion",
        "T-LEAP w/o (CL+Syn)",
        # "T-LEAP w/o (Syn)",
        # "T-LEAP w/o (CL)",
        "T-LEAP w/o theory",
        "T-LEAP",

    ],

    "AUC": [

        f"{np.mean(xgb_auc):.4f} ± {np.std(xgb_auc):.4f}",
        f"{np.mean(lstm_auc):.4f} ± {np.std(lstm_auc):.4f}",
        f"{np.mean(bert_auc):.4f} ± {np.std(bert_auc):.4f}",
        f"{np.mean(model4_auc):.4f} ± {np.std(model4_auc):.4f}",
        # f"{np.mean(model5_auc):.4f} ± {np.std(model5_auc):.4f}",
        # f"{np.mean(model6_auc):.4f} ± {np.std(model6_auc):.4f}",

        f"{np.mean(model8_auc):.4f} ± {np.std(model8_auc):.4f}",
        f"{np.mean(model7_auc):.4f} ± {np.std(model7_auc):.4f}",

    ],

    "Accuracy": [

        f"{np.mean(xgb_acc):.4f} ± {np.std(xgb_acc):.4f}",
        f"{np.mean(lstm_acc):.4f} ± {np.std(lstm_acc):.4f}",
        f"{np.mean(bert_acc):.4f} ± {np.std(bert_acc):.4f}",
        f"{np.mean(model4_acc):.4f} ± {np.std(model4_acc):.4f}",
        # f"{np.mean(model5_acc):.4f} ± {np.std(model5_acc):.4f}",
        # f"{np.mean(model6_acc):.4f} ± {np.std(model6_acc):.4f}",

        f"{np.mean(model8_acc):.4f} ± {np.std(model8_acc):.4f}",
        f"{np.mean(model7_acc):.4f} ± {np.std(model7_acc):.4f}"

    ],

    "F1": [

        f"{np.mean(xgb_f1):.4f} ± {np.std(xgb_f1):.4f}",
        f"{np.mean(lstm_f1):.4f} ± {np.std(lstm_f1):.4f}",
        f"{np.mean(bert_f1):.4f} ± {np.std(bert_f1):.4f}",
        f"{np.mean(model4_f1):.4f} ± {np.std(model4_f1):.4f}",
        # f"{np.mean(model5_f1):.4f} ± {np.std(model5_f1):.4f}",
        # f"{np.mean(model6_f1):.4f} ± {np.std(model6_f1):.4f}",
        f"{np.mean(model8_f1):.4f} ± {np.std(model8_f1):.4f}",
        f"{np.mean(model7_f1):.4f} ± {np.std(model7_f1):.4f}"
        

    ]

})

print("\n")
print("=" * 75)
print("5-Fold Cross Validation Results")
print("=" * 75)
print(results_.to_string(index=False))



5-Fold Cross Validation Results
                                        Model             AUC        Accuracy              F1
                                      XGBoost 0.7587 ± 0.0142 0.6840 ± 0.0132 0.6480 ± 0.0289
                            T-LEAP w/o Prompt 0.6943 ± 0.0173 0.6345 ± 0.0148 0.5764 ± 0.0397
T-LEAP w/o Align(ment)-cross-attention fusion 0.7607 ± 0.0239 0.7065 ± 0.0075 0.7205 ± 0.0369
                          T-LEAP w/o (CL+Syn) 0.7652 ± 0.0240 0.7160 ± 0.0208 0.7335 ± 0.0306
                            T-LEAP w/o theory 0.7513 ± 0.0267 0.6885 ± 0.0231 0.6832 ± 0.0330
                                       T-LEAP 0.7916 ± 0.0098 0.7465 ± 0.0171 0.7435 ± 0.0186


# 数据量为9K时, 效果很好, 但运算时间很慢